In [ ]:
# Linalg
import jax.numpy as np
import numpy as onp

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm
from matplotlib import cm
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d import Axes3D

# Data handling
import h5py as hf
import pandas as pd

# Analysis
from atomcorr.analysis.onsager_coefficients import OnsagerCoefficients

# Helper functions
from rich.progress import track

In [ ]:

main_prefixes = [
    "/data/stovey/mario-big-scans/projects/2024-03-08-Lymburn_default_noise_strength_alignment_strength_scan_radius_1.0",
    "/data/stovey/mario-big-scans/projects/2024-03-08-Lymburn_default_noise_strength_alignment_strength_scan_radius_4.0",
    "/data/stovey/mario-big-scans/projects/2024-03-08-Lymburn_slow_noise_strength_alignment_strength_scan_radius_1.0",
    "/data/stovey/mario-big-scans/projects/2024-03-08-Lymburn_slow_noise_strength_alignment_strength_scan_radius_4.0"
]

In [ ]:
def normalize_covariance_matrix(covariance_matrix: np.ndarray):
    """
    Method for normalizing a covariance matrix.
    Returns
    -------
    normalized_covariance_matrix : np.ndarray
            A normalized covariance matrix, i.e, the matrix given if all of its inputs
            had been normalized.
    """
    order = np.shape(covariance_matrix)[0]

    diagonals = np.diagonal(covariance_matrix)

    repeated_diagonals = np.repeat(diagonals[None, :], order, axis=0)

    normalizing_matrix = np.sqrt(repeated_diagonals * repeated_diagonals.T)

    return covariance_matrix / normalizing_matrix

In [ ]:
params = np.linspace(0, 24, 25, dtype=int)
calculator = OnsagerCoefficients()
entropies = {}
traces = {}
corr_matrix = {}

for main_prefix in main_prefixes:
    corr_data = pd.read_csv(f"{main_prefix}/aggregate.time_avg.ensemble_avg.lymburn_correlation_coefficient_test.csv")
    corr_matrix[main_prefix] = corr_data["observable"].to_numpy()

    entropies[main_prefix] = []
    traces[main_prefix] = []
    for param in track(params):
        prefix = f"{main_prefix}/parameter-combination-{param}/seed-0"
        

        with hf.File(f"{prefix}/simulation_output_train.h5", "r") as db:
            data = db["agent_observables"]["agent_velocity"][:]

            full_correlation = calculator._compute_correlation_matrix(
                data, 
                data, 
                correlation_time=1000,
                data_range=300
            )
            eigs = np.clip(np.linalg.eigh(full_correlation)[0], 1e-8, None)
            eigs /= eigs.sum()

            entropy = (-eigs * np.log(eigs)).sum()
            entropies[main_prefix].append(entropy)
            traces[main_prefix].append(np.trace(full_correlation))


In [ ]:
noise_strength = np.array([0.01, 0.1, 1.0, 10.0, 100.0])
alignment_strength = np.array([0.001, 0.01, 0.1, 1.0, 10.0])

In [ ]:
x, y = np.meshgrid(noise_strength, alignment_strength)

In [ ]:
fig = plt.figure(figsize=(18, 6))  # Adjust the figure size to your liking

# Create subplots
loss_ax = fig.add_subplot(1, 3, 1, projection='3d')
entropy_ax = fig.add_subplot(1, 3, 2, projection="3d")
trace_ax = fig.add_subplot(1, 3, 3, projection="3d")

# Prepare colorbar
# Normalization for the colorbar
norm = Normalize(vmin=-1, vmax=1)

# Create a ScalarMappable for the colorbar
sm = cm.ScalarMappable(cmap='viridis', norm=norm)
sm.set_array([])  # You have to set the array for the ScalarMappable

# Reshape the data
entropy = np.reshape(np.array(entropies), (4, 5, 5))
traces = np.reshape(np.array(traces), (4, 5, 5))
corr_data = np.reshape(np.array(corr_matrix), (4, 5, 5))

for e, t, c in zip(entropy, traces, corr_data):
    combi = np.concatenate(
        (
            x[:, :, None],
            y[:, :, None],
            e[:, :, None],
            t[:, :, None],
            c[:, :, None]
        ),
        axis=-1
    )

    # Loss data
    coords = np.concatenate(np.concatenate((combi[:, :, 0:1], combi[:, :, 1:2], combi[:, :, 4:]), axis=-1))
    loss_ax.scatter(np.log10(coords[:, 0]), np.log10(coords[:, 1]), coords[:, 2], c="blue")

    # Set axis labels
    loss_ax.set_xlabel('Noise')
    loss_ax.set_ylabel('Lymburn Coupling')
    loss_ax.set_zlabel('Correlation Score')

    # Set axis scales
    # loss_ax.set_xscale("log")
    # loss_ax.set_yscale("log")


    # Entropy data, colored by loss
    coords = np.concatenate(np.concatenate((combi[:, :, 0:1], combi[:, :, 1:2], combi[:, :, 2:]), axis=-1))
    c = combi[:, :, 4]  # Loss
    entropy_ax.scatter(np.log10(coords[:, 0]), np.log10(coords[:, 1]), coords[:, 2], c=c, cmap='viridis')

    # Set axis labels
    entropy_ax.set_xlabel('Noise')
    entropy_ax.set_ylabel('Lymburn Coupling')
    entropy_ax.set_zlabel('Entropy')


    # Trace data, colored by loss
    coords = np.concatenate(np.concatenate((combi[:, :, 0:1], combi[:, :, 1:2], combi[:, :, 3:]), axis=-1))
    c = combi[:, :, 4]  # Loss
    trace_ax.scatter(np.log10(coords[:, 0]), np.log10(coords[:, 1]), np.log10(coords[:, 2]), c=c, cmap='viridis')

    # Set axis labels
    trace_ax.set_xlabel('Noise')
    trace_ax.set_ylabel('Lymburn Coupling')
    trace_ax.set_zlabel('Trace')

# Add colorbar
cbar_ax = fig.add_axes([0.4, 0.85, 0.5, 0.01])  # Example: right of the plots, thin, and tall
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Correlation Coefficient')

# plt.tight_layout()
plt.savefig('analysis.png')  # Save the figure to a file
plt.show()

In [ ]:
fig, ax = plt.subplots(4, 3, figsize=(15, 10))

for i, pref in enumerate(main_prefixes):
    e = np.reshape(np.array(entropies[pref]), (5, 5))
    t = np.reshape(np.array(traces[pref]), (5, 5))
    c = np.reshape(np.array(corr_matrix[pref]), (5, 5))

    cmap = 'RdBu'
    norm = LogNorm()

    mesh0 = ax[i][0].pcolormesh(np.log10(x), np.log10(y), c, cmap=cmap)

    fig.colorbar(mesh0, ax=ax[i][0])

    mesh1 = ax[i][1].pcolormesh(np.log10(x), np.log10(y), e, cmap=cmap)
    fig.colorbar(mesh1, ax=ax[i][1])

    mesh2 = ax[i][2].pcolormesh(np.log10(x), np.log10(y), t, cmap=cmap, norm=norm)
    fig.colorbar(mesh2, ax=ax[i][2])


for i in range(4):
    ax[i][0].set_ylabel("Lymburn Coupling")

for i in range(3):
    ax[3][i].set_xlabel("Noise Strength")

ax[0][0].set_title("Pearson Correlation")
ax[0][1].set_title("Entropy")
ax[0][2].set_title("Trace")

plt.savefig("Heatmaps.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.plot(traces)
ax.set_yscale("log")

ax2 = ax.twinx()

ax2.plot(corr_matrix, "r")

ax.set_xlabel("parameter")


ax.set_ylabel("Trace")
ax2.set_ylabel("Correlation Coefficient")

# plt.savefig("traec-diagram.pdf")

plt.show()


In [ ]:
fig, ax = plt.subplots()

plt.scatter(traces, corr_matrix, c=entropies)

# plt.savefig("traec-diagram.pdf")
plt.xscale("log")

plt.colorbar()
plt.show()

In [ ]:
fig, ax = plt.subplots()

plt.scatter(entropies, corr_matrix, c=traces, norm=LogNorm())

# plt.savefig("traec-diagram.pdf")
plt.colorbar()
plt.show()

In [ ]:
fig, ax = plt.subplots()

plt.scatter(traces, entropies, c=corr_matrix)

# plt.savefig("traec-diagram.pdf")

plt.colorbar()
plt.xscale("log")
plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.plot(entropies)
ax2 = ax.twinx()

ax2.plot(corr_matrix, "r")

ax.set_xlabel("parameter")


ax.set_ylabel("Entropy")
ax2.set_ylabel("Correlation Coefficient")


# plt.savefig("entropy-diagram.pdf")
plt.show()
